<a href="https://colab.research.google.com/github/MayerT1/PIPECAST/blob/main/step3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

In this module, we will take a sample image from ICEYE and run produce a binary water map. We will do this by following these steps

1. Obtain ICEYE image
2. Speckle Filter the image
3. Psuedo Terrain correct the image
4. Determine initial threshold through sampling points that have recurring water using JRC
5. Run HYDRAFloods Edge Otsu algorithm

# Part 1: Housekeeping and obtaining ICEYE image

In [ ]:
my_gee_folder = 'users/mickymags/watchduty/'

In [ ]:
!pip install ipyleaflet==0.18.2 geemap hydrafloods     # Install hydrafloods and its relevant dependencies
!pip install geemap

In [ ]:
from hydrafloods import corrections
import hydrafloods as hf
import ee
import geemap
import matplotlib.pyplot as plt

In [ ]:
ee.Authenticate()
ee.Initialize(project = 'servir-sco-assets')

In [ ]:
imgcollection = ee.ImageCollection('projects/servir-sco-assets/assets/watchduty/iceye_test')
ic1 = imgcollection.filterDate('2020-03-31', '2020-04-01').first();

iceye_scale = ic1.projection().nominalScale().getInfo()
iceye_scale

In [ ]:
ic1.bandNames().getInfo()

In [ ]:
#aoi = ee.FeatureCollection('users/mickymags/iceye_aoi')
aoi = ee.FeatureCollection('projects/servir-sco-assets/assets/watchduty/memphis_aoi')
aoi_geom = aoi.geometry()

# Get the coordinates of the center of the AOI for mapping purposes
aoi_centroid = aoi.geometry().centroid()             # Get the center of the AOI
lon = aoi_centroid.coordinates().get(0).getInfo()    # Extract the longitude from the centroid
lat = aoi_centroid.coordinates().get(1).getInfo()    # Extract the latitude from the centroid

#my_geom_centroid = my_geom.centroid()
#lon = my_geom_centroid.coordinates().get(0).getInfo()    # Extract the longitude from the centroid
#lat = my_geom_centroid.coordinates().get(1).getInfo()    # Extract the latitude from the centroid

#my_geom = ee.Geometry.Polygon([[[4.09920685428026, 51.96369505639564], [4.1681386239307825, 51.97807068435677],
                                #[4.191350812483022, 51.93549196538367], [4.122433887623417, 51.92112965885514]]])

In [ ]:
aoi_geom.getInfo()

In [ ]:
iceye = hf.Dataset(
    region = aoi_geom,               #my_geom
    start_time = '2020-03-30',
    end_time = '2020-04-01',
    asset_id = 'projects/servir-sco-assets/assets/watchduty/iceye_test' # change this to iceye image collection if this doesn't work
)

test histrogram

In [ ]:
iceye_scale

In [ ]:
# Load your SAR image from assets
asset_id = "projects/servir-sco-assets/assets/watchduty/iceye_test/memphis_0918"
image = ee.Image(asset_id)

# Get band names
bands = image.select('VV').bandNames().getInfo()
print("Bands in image:", bands)

# Create a histogram for each band
def plot_histograms(image, bands,  aoi_descriptor, region, scale=3.46325, n_bins=4096):
    """Plot histograms for each band of an ee.Image."""
    fig, axes = plt.subplots(len(bands), 1, figsize=(8, 4*len(bands)))

    if len(bands) == 1:
        axes = [axes]

    for i, band in enumerate(bands):
        # Reduce region to histogram
        hist_dict = image.select(band).reduceRegion(
            reducer=ee.Reducer.histogram(maxBuckets=n_bins), #maxBuckets=n_bins),
            geometry=region,
            scale=scale,

            bestEffort=True
        ).get(band).getInfo()

        if hist_dict is not None:
            counts = hist_dict['histogram']
            bucketMeans = hist_dict['bucketMeans']

            axes[i].bar(bucketMeans, counts, width=(bucketMeans[1]-bucketMeans[0]))
            axes[i].set_title(f"Histogram of {band} for {aoi_descriptor}")
            axes[i].set_xlabel("Pixel values")
            axes[i].set_ylabel("Frequency")
            axes[i].set_xlim(-40, 10) #0, 250
        else:
            axes[i].set_title(f"{band}: No data")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_histograms(image, bands, 'Memphis', aoi_geom)

As we can see, we have a very unimodal histogram. This does not bode well for our thresholding algorithm, which attempts to seek a minimum between a bimodal histogram by maximizing intraclass variance

In [ ]:
iceye.n_images

In [ ]:
iceye_med = iceye.collection.median()
#iceye_mul = iceye_med.multiply(1e5)
iceye_vp = {
    'bands': 'VV', #'b1',
    'min': 0,
    'max': 200
}

Get a sentinel-1 image from the same time

In [ ]:
s1 = ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi_geom).filterDate('2020-03-30', '2020-04-01').first().clip(aoi_geom)

s1_vp = {
    'bands': 'VV',
    'min': -20,
    'max': 0
}

In [ ]:
plot_histograms(s1, bands, 'Memphis', aoi_geom)

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 8)
Map.addLayer(aoi_geom, {}, 'Area of Interest')
Map.addLayer(s1, s1_vp, 'Sentinel-1')
Map.addLayer(iceye_med, iceye_vp, 'ICEYE')

Map.addLayerControl()
Map

# Part 2: Speckle Filtering

In [ ]:
iceye_med

In [ ]:
ic_lee = iceye.apply_func(hf.lee_sigma)
ic_gamma = iceye.apply_func(hf.gamma_map)
ic_refined = iceye.apply_func(hf.refined_lee)

In [ ]:
ic_lee_med = ic_lee.collection.median()
ic_gamma_med = ic_gamma.collection.median()
ic_refined_med = ic_refined.collection.median()

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 8)
Map.addLayer(iceye_med, iceye_vp, 'ICEYE')
Map.addLayer(ic_lee_med, iceye_vp, 'ICEYE Lee Sig')
Map.addLayer(ic_gamma_med, iceye_vp, 'ICEYE GAMMA')
Map.addLayer(ic_refined_med, iceye_vp, 'ICEYE REFINED')


Map.addLayerControl()
Map

In [ ]:
geemap.ee_export_image_to_drive(image = iceye_med,
                                #region = export_aoi,
                                folder = 'watchduty',
                                description = 'iceye_raw_img_test',
                                scale=30,
                                maxPixels = 1e13)

geemap.ee_export_image_to_drive(image = ic_lee_med,
                                #region = export_aoi,
                                folder = 'watchduty',
                                description = 'iceye_lee_img_',
                                scale=30,
                                maxPixels = 1e13)

geemap.ee_export_image_to_drive(image = ic_gamma_med,
                                #region = export_aoi,
                                folder = 'watchduty',
                                description = 'iceye_gamma_img_test',
                                scale=30,
                                maxPixels = 1e13)

geemap.ee_export_image_to_drive(image = ic_refined_med,
                                #region = export_aoi,
                                folder = 'watchduty',
                                description = 'iceye_refined_img_test',
                                scale=30,
                                maxPixels = 1e13)

In [ ]:
plot_histograms(iceye_med, bands, 'Memphis Raw Image', aoi_geom)

In [ ]:
plot_histograms(ic_lee_med, bands, 'Memphis Lee Filtered Image', aoi_geom, n_bins = 100)

In [ ]:
plot_histograms(ic_gamma_med, bands, 'Memphis Gamma Filtered Image', aoi_geom, n_bins = 100)

In [ ]:
plot_histograms(ic_refined_med, bands, 'Memphis Refined Lee Filtered Image', aoi_geom, n_bins = 100)

# Part 3: Psuedo terrain correction

In [ ]:
#elv = ee.Image("JAXA/ALOS/AW3D30/V2_2").select("AVE_DSM")
elv = ee.ImageCollection("USGS/3DEP/10m_collection").filterBounds(aoi_geom).mosaic().clip(aoi_geom)

In [ ]:
print(elv.getInfo())

In [ ]:
#iceye terrain corrected
ic_tc = ic_gamma.apply_func(corrections.slope_correction, elevation=elv, buffer=30)

# median image
ictc_med = ic_tc.collection.median()

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 12)
Map.addLayer(aoi_geom, {}, 'Area of Interest')
Map.addLayer(iceye_med, iceye_vp, 'ICEYE')
Map.addLayer(ic_gamma_med, iceye_vp, 'ICEYE Speckle')
Map.addLayer(ictc_med, iceye_vp, 'Speckle + TC')

Map.addLayerControl()
Map

# Part 4: Initial Threshold determination

In [ ]:
iceye_sampled = ic_lee_med.sample(aoi, numPixels = 1000, scale = iceye_scale).getInfo()
features = iceye_sampled['features']
first_feature = features[0]
first_feature['properties']['VV']

vals = []

for j in range(len(features)):
  feat_of_int = features[j]
  vals.append(feat_of_int['properties']['VV'])

In [ ]:
import numpy as np

In [ ]:
np_vals = np.array(vals)
np_vals.max()

In [ ]:
np_vals.mean()

# Part 5: Run Edge Otsu Thresholding Algorithm

In [ ]:
export_aoi = ee.Geometry.Rectangle([-90.225, 35.22, -90.01, 35.48])

In [ ]:
edge = hf.edge_otsu(
    iceye_med, #iceye_med
    band = 'VV',
    region = export_aoi,
    edge_buffer = 30,
    initial_threshold = 40,
    thresh_no_data = 30,
    scale = 30
)

In [ ]:
bmax = hf.bmax_otsu(ic_gamma_med, band = 'VV', region = aoi, scale = 30, thresh_no_data = 60)

In [ ]:
water_vp = {
    'palette': ['000000', 'add8e6'],
    'min': 0,
    'max': 1
}

In [ ]:
iceye_scale

In [ ]:
# Edge algorithm using the refined-lee-sigma speckle filtering method
edge_refined = hf.edge_otsu(
    ic_refined_med, #iceye_med
    band = 'VV',
    region = export_aoi,
    edge_buffer = 300,
    initial_threshold = 92,
    thresh_no_data = 90,
    scale = 3.463225
)

In [ ]:
planet = ee.Image('projects/servir-sco-assets/assets/watchduty/memphis_planet')

pl_vp = {
    'bands': ['b3', 'b2', 'b1'],
    'min': 0,
    'max': 3000
}

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 12)
Map.addLayer(aoi, {}, 'Area of Interest')
Map.addLayer(iceye_med, iceye_vp, 'ICEYE')
#Map.addLayer(ic_lee_med, iceye_vp, 'ICEYE Lee')
#Map.addLayer(ic_refined_med, iceye_vp, 'ICEYE Refined');
Map.addLayer(edge, water_vp, 'Edge')
Map.addLayer(planet, pl_vp, 'Plantet')
#Map.addLayer(edge_refined, water_vp, 'Edge Refined')
#Map.addLayer(bmax, water_vp, 'Bmax')
Map.addLayerControl()
Map

In [ ]:
#

In [ ]:
iceye_scale

In [ ]:
geemap.ee_export_image_to_drive(image = edge_refined,
                                region = export_aoi,
                                folder = 'watchduty',
                                description = 'hf_edge_refined_v2',
                                scale=3.463225,
                                maxPixels = 1e13)

In [ ]:
geemap.ee_export_image_to_drive(image = edge,
                                region = export_aoi,
                                folder = 'watchduty',
                                description = 'hf_test_edge',
                                scale=3.463225,
                                maxPixels = 1e13)

geemap.ee_export_image_to_drive(image = edge,
                                region = export_aoi,
                                folder = 'watchduty',
                                description = 'hf_test_bmax',
                                scale=3.463225,
                                maxPixels = 1e13)


# Part 6: Workflow demonstration

This section is meant to demonstrate the entire HYDRAFloods workflow within one code cell

In [ ]:
# Get ICEYE dataset
iceye = hf.Dataset(
    region = aoi_geom,                                         # The region of interest (of type Earth Engine geometry)
    start_time = '2020-03-30',                                 # The start date of interest
    end_time = '2020-04-01',                                   # The end date of interest
    asset_id = 'projects/my_gee_project/my_iceye_images'       # The GEE Asset ID of the ICEYE imagery
)

# Speckle Filtering
iceye_speckle = iceye.apply_func(hf.lee_sigma)                                  # Apply the Lee Sigma speckle filter to the above dataset
ic_tc = iceye_speckle.apply_func(corrections.slope_correction, elevation=elv,
                                 buffer=30).collection().median()               # Apply a terrain correction
# Histogram Thresholding
edge = hf.edge_otsu(                 # Apply the Edge Otsu algorithm
    ic_tc,                          # Using the speckle-filtered, terrain-corrected Iceye Image
    band = 'VV',                     # On just the Vertically polarized band
    region = aoi_geom,               # Over the aforementioned region of interest
    edge_buffer = 30,                # Buffer the edges identified by the Canny algorithm by 30 meters on either side to sample
    initial_threshold = 40,          # The initial threshold value the edge otsu will try to improve upon
    thresh_no_data = 30,             # The threshold to use for pixels masked by terrain correction
    scale = 3                        # The scale in meters to use
)